In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set working directory
import os
os.chdir('/content/drive/MyDrive/brain-tumor-project/notebooks')

# Verify
!ls ../data/classification/train/

# 01 - Exploratory Data Analysis (EDA)

Comprehensive analysis of the merged brain tumor MRI dataset.

**Dataset:** Merged from 3 Kaggle sources | **Classes:** Glioma, Meningioma, No Tumor, Pituitary

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

BASE_DIR = os.path.dirname(os.getcwd())
sys.path.append(BASE_DIR)
DATA_DIR = os.path.join(BASE_DIR, "data", "classification")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
CLASS_LABELS = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]

print(f"Data directory: {DATA_DIR}")

## 1. Dataset Overview

In [ ]:
def count_images(base_dir, classes):
    counts = {}
    for cls in classes:
        cls_dir = os.path.join(base_dir, cls)
        if os.path.exists(cls_dir):
            counts[cls] = len([f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        else:
            counts[cls] = 0
    return counts

train_counts = count_images(TRAIN_DIR, CLASSES)
test_counts = count_images(TEST_DIR, CLASSES)

df_summary = pd.DataFrame({
    "Class": CLASS_LABELS,
    "Training": [train_counts[c] for c in CLASSES],
    "Testing": [test_counts[c] for c in CLASSES],
})
df_summary["Total"] = df_summary["Training"] + df_summary["Testing"]
df_summary["Train %"] = (df_summary["Training"] / df_summary["Training"].sum() * 100).round(1)

print("\n" + "="*60)
print("DATASET SUMMARY")
print("="*60)
print(df_summary.to_string(index=False))
print(f"\nTotal Training:  {df_summary['Training'].sum()}")
print(f"Total Testing:   {df_summary['Testing'].sum()}")
print(f"Grand Total:     {df_summary['Total'].sum()}")

## 2. Class Distribution Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4"]

# Training distribution
bars = axes[0].bar(CLASS_LABELS, [train_counts[c] for c in CLASSES], color=colors, edgecolor="white", linewidth=1.5)
axes[0].set_title("Training Set Distribution", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Number of Images", fontsize=12)
for bar, count in zip(bars, [train_counts[c] for c in CLASSES]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, str(count), ha="center", fontweight="bold")

# Test distribution
bars = axes[1].bar(CLASS_LABELS, [test_counts[c] for c in CLASSES], color=colors, edgecolor="white", linewidth=1.5)
axes[1].set_title("Test Set Distribution", fontsize=14, fontweight="bold")
for bar, count in zip(bars, [test_counts[c] for c in CLASSES]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(count), ha="center", fontweight="bold")

# Pie chart
axes[2].pie([train_counts[c] + test_counts[c] for c in CLASSES], labels=CLASS_LABELS, colors=colors, autopct="%1.1f%%", startangle=90)
axes[2].set_title("Overall Distribution", fontsize=14, fontweight="bold")

plt.suptitle("Brain Tumor MRI Dataset - Class Distribution", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "eda_class_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. Sample Images from Each Class

In [ ]:
fig, axes = plt.subplots(4, 6, figsize=(18, 12))

for row, cls in enumerate(CLASSES):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    if not os.path.exists(cls_dir):
        continue
    images = os.listdir(cls_dir)[:6]
    for col, img_name in enumerate(images):
        img = Image.open(os.path.join(cls_dir, img_name)).convert("RGB")
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(CLASS_LABELS[row], fontsize=14, fontweight="bold", rotation=0, labelpad=80)

plt.suptitle("Sample MRI Images from Each Class", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "eda_sample_images.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Image Dimension Analysis

In [ ]:
widths, heights = [], []

for cls in CLASSES:
    cls_dir = os.path.join(TRAIN_DIR, cls)
    if not os.path.exists(cls_dir):
        continue
    for img_name in tqdm(os.listdir(cls_dir)[:500], desc=f"Analyzing {cls}"):
        try:
            img = Image.open(os.path.join(cls_dir, img_name))
            w, h = img.size
            widths.append(w)
            heights.append(h)
        except:
            pass

print(f"Width  - Min: {min(widths)}, Max: {max(widths)}, Mean: {np.mean(widths):.0f}")
print(f"Height - Min: {min(heights)}, Max: {max(heights)}, Mean: {np.mean(heights):.0f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(widths, bins=50, color="#FF6B6B", alpha=0.7, edgecolor="white")
axes[0].set_title("Width Distribution", fontsize=14, fontweight="bold")
axes[0].axvline(np.mean(widths), color="red", linestyle="--", label=f"Mean: {np.mean(widths):.0f}")
axes[0].legend()

axes[1].hist(heights, bins=50, color="#4ECDC4", alpha=0.7, edgecolor="white")
axes[1].set_title("Height Distribution", fontsize=14, fontweight="bold")
axes[1].axvline(np.mean(heights), color="teal", linestyle="--", label=f"Mean: {np.mean(heights):.0f}")
axes[1].legend()

axes[2].scatter(widths, heights, alpha=0.3, s=10, c="#45B7D1")
axes[2].set_title("Width vs Height", fontsize=14, fontweight="bold")
axes[2].set_xlabel("Width")
axes[2].set_ylabel("Height")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "eda_dimensions.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Pixel Intensity Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors_map = {"glioma": "#FF6B6B", "meningioma": "#4ECDC4", "notumor": "#45B7D1", "pituitary": "#96CEB4"}

for idx, cls in enumerate(CLASSES):
    ax = axes[idx // 2, idx % 2]
    cls_dir = os.path.join(TRAIN_DIR, cls)
    if not os.path.exists(cls_dir):
        continue
    intensities = []
    for img_name in os.listdir(cls_dir)[:100]:
        try:
            img = np.array(Image.open(os.path.join(cls_dir, img_name)).convert("L"))
            intensities.extend(img.flatten().tolist())
        except:
            pass
    ax.hist(intensities, bins=256, range=(0, 255), color=colors_map[cls], alpha=0.7, density=True)
    ax.set_title(f"{CLASS_LABELS[idx]} - Pixel Intensity", fontsize=13, fontweight="bold")
    ax.set_xlabel("Pixel Value")
    ax.set_ylabel("Density")
    ax.set_xlim(0, 255)

plt.suptitle("Pixel Intensity Distributions by Class", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "eda_pixel_intensity.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Data Augmentation Preview

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

cls_dir = os.path.join(TRAIN_DIR, "glioma")
if os.path.exists(cls_dir):
    sample_path = os.path.join(cls_dir, os.listdir(cls_dir)[0])
    sample_img = np.array(Image.open(sample_path).convert("RGB").resize((224, 224)))
    sample_img = sample_img.reshape((1,) + sample_img.shape)

    datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.15,
        zoom_range=0.2,
        horizontal_flip=True,
        brightness_range=[0.8, 1.2],
        fill_mode="nearest"
    )

    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes[0, 0].imshow(sample_img[0])
    axes[0, 0].set_title("Original", fontsize=12, fontweight="bold")
    axes[0, 0].axis("off")

    for i, batch in enumerate(datagen.flow(sample_img, batch_size=1)):
        if i >= 9:
            break
        row = (i + 1) // 5
        col = (i + 1) % 5
        axes[row, col].imshow(batch[0].astype(np.uint8))
        axes[row, col].set_title(f"Augmented {i+1}", fontsize=12)
        axes[row, col].axis("off")

    plt.suptitle("Data Augmentation Examples (Glioma)", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, "eda_augmentation.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 7. Key Findings Summary

| Metric | Value |
|--------|-------|
| Total Images | *Run cells above* |
| Classes | 4 (Glioma, Meningioma, No Tumor, Pituitary) |
| Image Size | Resized to 224x224 |
| Merged From | 3 Kaggle datasets |
| Deduplication | Perceptual hashing |

**Next Steps:** Proceed to model training notebooks (02-11)